# Concat + MLP fusion: remaining 25 seeds (matched Phase 3 splits)

This notebook trains **concat + MLP** fusion on the **25** Phase-3 seeds that are not in the original 5-seed Colab list `[13, 17, 53, 309, 347]`. Seeds are read from `experiments/phase3_robustness/metrics/seeds_list.txt`.

**Splits:** Load `data/splits/seed_<id>/{train,val,test}.csv`. Those CSVs must come from the **same** `df_full` construction and two-step stratified `train_test_split` as Phase 3 (same `random_state` as the seed). If any folder is missing, generate the remaining 25 from the repo root with:

`python scripts/export_phase3_splits_to_csv.py`

Use `--all-from-seeds-list` to refresh all 30 seeds (including the five Colab seeds) from `caption_dataset_final_full.csv`.

**Protocol:** Same as `notebooks/robustness/Phase3_Robustness_Experiments.ipynb` (stratified `train_test_split` on the full caption CSV, frozen CLIP + BERT, hyperparameters, early stopping on val macro-F1), except the fusion module is **concat(512+768) -> MLP -> 512**.

**Outputs:** `experiments/concat_mlp_robustness/metrics/experiments/seed_*_results.json`. Copy your 5 Colab JSONs for seeds 13,17,53,309,347 into the same folder to complete all 30 seeds.


In [1]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
import json
import os

_walk = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_walk, "experiments")) and os.path.isdir(os.path.join(_walk, "data")):
        PROJECT_ROOT = _walk
        break
    _walk = os.path.dirname(_walk)
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())
os.chdir(PROJECT_ROOT)

from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import warnings
import time
from datetime import datetime
from urllib.parse import unquote

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

LEARNING_RATE = 5e-5
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20
MODEL_INIT_SEED = 42

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

PHASE3_SEEDS_FILE = os.path.join(
    PROJECT_ROOT, "experiments", "phase3_robustness", "metrics", "seeds_list.txt"
)
with open(PHASE3_SEEDS_FILE, "r") as f:
    _txt = f.read()
ALL_PHASE3_SEEDS = [int(x) for x in re.findall(r"Seed (\d+)", _txt)]
if len(ALL_PHASE3_SEEDS) != 30:
    raise ValueError("Expected 30 seeds in " + PHASE3_SEEDS_FILE)

SEEDS_ALREADY_5 = {13, 17, 53, 309, 347}
SEEDS = [s for s in ALL_PHASE3_SEEDS if s not in SEEDS_ALREADY_5]
assert len(SEEDS) == 25

EXPERIMENT_ROOT = os.path.join(PROJECT_ROOT, "experiments", "concat_mlp_robustness")
METRICS_DIR = os.path.join(EXPERIMENT_ROOT, "metrics")
ARTIFACTS_DIR = os.path.join(EXPERIMENT_ROOT, "artifacts")
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(os.path.join(METRICS_DIR, "experiments"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "learning_curves"), exist_ok=True)

_seeds_log = os.path.join(METRICS_DIR, "concat_mlp_remaining25_seeds.txt")
with open(_seeds_log, "w") as f:
    f.write("Remaining 25 seeds (Phase 3 list minus 13,17,53,309,347)\n")
    for i, s in enumerate(SEEDS, 1):
        f.write("{:2d}. Seed {}\n".format(i, s))

print("SEEDS (25):", SEEDS)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)


/home/ding-zhang/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
SEEDS (25): [14, 16, 45, 48, 58, 72, 102, 112, 115, 120, 126, 141, 215, 217, 259, 280, 288, 303, 328, 333, 360, 367, 378, 380, 457]
EXPERIMENT_ROOT: /home/ding-zhang/Dongmei/DATA255/Project/experiments/concat_mlp_robustness


## Load full dataset


In [2]:
print("Loading caption dataset...")
df = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "processed", "caption_dataset_final_full.csv"))
df_success = df[df["status"] == "success"].copy()
df_success["style"] = df_success["style"].str.strip()
all_styles = sorted(df_success["style"].unique())
style_to_idx = {style: idx for idx, style in enumerate(all_styles)}
num_classes = len(all_styles)
df_full = df_success.copy()
captions_dict = {row["image_path"]: row["caption"] for _, row in df_full.iterrows()}
print("Full dataset size:", len(df_full), "classes:", num_classes)


Loading caption dataset...
Full dataset size: 13230 classes: 14


## Load encoders


In [3]:
import clip
from transformers import AutoTokenizer, AutoModel

print("Loading CLIP...")
clip_model, _ = clip.load("ViT-B/32", device=device)
clip_model.eval()

print("Loading BERT tokenizer/model...")
fashionbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
fashionbert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
fashionbert_model.eval()
print("Done.")


Loading CLIP...
Loading BERT tokenizer/model...


2026-05-12 17:01:06.854433: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 17:01:06.876042: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Done.


2026-05-12 17:01:07.539124: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Dataset, Concat+MLP fusion, classifier


In [4]:
class FashionMultiModalDataset(Dataset):
    """Same as Phase 3."""

    def __init__(self, df, captions_dict, style_to_idx, transform=None, base_dir=None):
        self.df = df.reset_index(drop=True)
        self.captions_dict = captions_dict
        self.style_to_idx = style_to_idx
        self.transform = transform
        self.base_dir = base_dir
        self.valid_indices = []
        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            image_path = row["image_path"]
            if base_dir and not os.path.isabs(image_path):
                _rel = str(image_path).replace("\\", "/")
                if _rel.startswith("dataset/") and not os.path.isdir(os.path.join(base_dir, "dataset")):
                    image_path = os.path.join(base_dir, "data", "raw dataset", _rel[len("dataset/"):])
                else:
                    image_path = os.path.join(base_dir, image_path)
            if "%" in image_path:
                path_parts = image_path.split("/")
                decoded_parts = [unquote(part) if "%" in part else part for part in path_parts]
                image_path = "/".join(decoded_parts)
            if image_path in captions_dict or row["image_path"] in captions_dict:
                self.valid_indices.append(idx)

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]
        image_path = row["image_path"]
        if self.base_dir and not os.path.isabs(image_path):
            _rel = str(image_path).replace("\\", "/")
            if _rel.startswith("dataset/") and not os.path.isdir(os.path.join(self.base_dir, "dataset")):
                image_path = os.path.join(self.base_dir, "data", "raw dataset", _rel[len("dataset/"):])
            else:
                image_path = os.path.join(self.base_dir, image_path)
        if "%" in image_path:
            path_parts = image_path.split("/")
            decoded_parts = [unquote(part) if "%" in part else part for part in path_parts]
            image_path = "/".join(decoded_parts)
        try:
            if os.path.exists(image_path):
                image = Image.open(image_path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
            else:
                image = torch.zeros(3, 224, 224)
        except Exception:
            image = torch.zeros(3, 224, 224)
        caption = self.captions_dict.get(
            image_path, self.captions_dict.get(row["image_path"], "No caption available")
        )
        style = row["style"]
        label = self.style_to_idx[style]
        return {
            "image": image,
            "caption": caption,
            "label": label,
            "style": style,
            "image_path": image_path,
        }


class ConcatMLPFusion(nn.Module):
    """Concat CLIP + BERT [CLS], then 2-layer MLP to 512-d."""

    def __init__(self, visual_dim, textual_dim, hidden_dim=512):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(visual_dim + textual_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, visual_feat, textual_feat):
        return self.mlp(torch.cat([visual_feat, textual_feat], dim=-1))


class MultiModalFashionClassifier(nn.Module):
    def __init__(self, clip_model, fashionbert_model, num_classes, dropout=0.5, visual_dim=512, textual_dim=768):
        super().__init__()
        self.clip_model = clip_model
        self.fashionbert_model = fashionbert_model
        self.fusion = ConcatMLPFusion(visual_dim, textual_dim)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, images, captions):
        with torch.no_grad():
            visual_features = self.clip_model.encode_image(images).float()
        with torch.no_grad():
            inputs = fashionbert_tokenizer(
                captions,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(device)
            outputs = self.fashionbert_model(**inputs)
            textual_features = outputs.last_hidden_state[:, 0, :]
        fused = self.fusion(visual_features, textual_features)
        return self.classifier(fused)

print("Model classes defined.")


Model classes defined.


In [ ]:
# DEBUG: verify image paths match FashionMultiModalDataset remapping (dataset/ -> data/raw dataset/)
import os
import numpy as np
import pandas as pd
from PIL import Image

def _resolve_image_path_debug(image_path, base_dir):
    """Must match FashionMultiModalDataset: relative dataset/... when no top-level dataset/ dir."""
    image_path = str(image_path)
    if base_dir and not os.path.isabs(image_path):
        _rel = image_path.replace("\\", "/")
        if _rel.startswith("dataset/") and not os.path.isdir(os.path.join(base_dir, "dataset")):
            return os.path.normpath(os.path.join(base_dir, "data", "raw dataset", _rel[len("dataset/"):]))
        return os.path.normpath(os.path.join(base_dir, image_path))
    return image_path

base_dir = globals().get("BASE_DIR", PROJECT_ROOT)
splits_root = os.path.join(PROJECT_ROOT, "data", "splits")
legacy_ds = os.path.join(base_dir, "dataset")
remapped_root = os.path.join(base_dir, "data", "raw dataset")

print("=== Image path sanity check ===")
print("BASE_DIR:", base_dir)
print("SPLITS_ROOT:", splits_root)
print("Top-level dataset/ exists:", os.path.isdir(legacy_ds))
print("data/raw dataset/ exists:", os.path.isdir(remapped_root))

probe_csv = os.path.join(splits_root, "seed_13", "train.csv")
if not os.path.isfile(probe_csv):
    probe_csv = os.path.join(splits_root, f"seed_{SEEDS[0]}", "train.csv")
print("Probe CSV:", probe_csv, "exists:", os.path.isfile(probe_csv))

df_probe = pd.read_csv(probe_csv, nrows=10)
n_ok = 0
for i, row in df_probe.iterrows():
    raw = row["image_path"]
    resolved = _resolve_image_path_debug(raw, base_dir)
    ok = os.path.isfile(resolved)
    n_ok += int(ok)
    im = Image.open(resolved).convert("RGB") if ok else None
    mean255 = float(np.array(im).mean()) if im is not None else float("nan")
    print(f"  row{i}: file_ok={ok} rgb_mean={mean255:.1f} (zeros/black broken ~0)")
    print(f"         raw: {raw}")
    print(f"         resolved: {resolved}")

print(f"\nSummary: {n_ok}/{len(df_probe)} files found. Expect ~10/10 and rgb_mean well above 0.")
if n_ok == 0:
    print("FAIL: no images resolved — training uses black tensors; fix BASE_DIR or path mapping.")
elif n_ok < len(df_probe):
    print("WARN: some paths missing — check CSV vs on-disk layout.")


## Training helpers


In [5]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(train_loader, desc="Training", leave=False):
        images = batch['image'].to(device)
        captions = batch['caption']
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(images, captions)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return total_loss / len(train_loader), 100. * correct / total

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            images = batch['image'].to(device)
            captions = batch['caption']
            labels = batch['label'].to(device)

            logits = model(images, captions)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0) if len(all_predictions) > 0 else 0.0

    return total_loss / len(val_loader), 100. * correct / total, all_predictions, all_labels, val_macro_f1

print("Training functions defined.")


Training functions defined.


## Experiment runner


In [6]:
def run_robustness_experiment(seed_value, seed_idx, df_full, captions_dict, style_to_idx,
                              clip_model, fashionbert_model, fashionbert_tokenizer,
                              num_classes, device, all_styles, base_dir):
    """
    Run a single robustness experiment with given seed
    
    Args:
        seed_value: Random seed value for data splitting (from 1-500)
        seed_idx: Index of seed in SEEDS list (1-30)
        df_full: Full dataset DataFrame
        captions_dict: Dictionary mapping image paths to captions
        style_to_idx: Dictionary mapping style names to indices
        clip_model, fashionbert_model: Pre-trained models
        fashionbert_tokenizer: BERT tokenizer
        num_classes: Number of classes
        device: Device
        all_styles: List of style names
        base_dir: Base directory for absolute paths
    
    Returns:
        Dictionary with all results and metrics
    """
    
    print(f"\n{'='*70}")
    print(f"Experiment {seed_idx}/{len(SEEDS)} (concat+MLP): Seed {seed_value}")
    print(f"{'='*70}")
    
    # Check if result already exists (resume capability)
    result_file = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if os.path.exists(result_file):
        print(f"  ⏭️  Result already exists, skipping...")
        with open(result_file, 'r') as f:
            return json.load(f)
    
    # Set fixed seed for model initialization (same for all experiments)
    torch.manual_seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)
    
    # Same splits as Phase 3 / Colab: CSVs under data/splits/seed_<seed>/
    split_dir = os.path.join(PROJECT_ROOT, "data", "splits", f"seed_{seed_value}")
    print(f"  Loading splits from {split_dir} ...")
    train_df = pd.read_csv(os.path.join(split_dir, "train.csv"))
    val_df = pd.read_csv(os.path.join(split_dir, "val.csv"))
    test_df = pd.read_csv(os.path.join(split_dir, "test.csv"))
    print(f"  Split sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
    
    # Image transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = FashionMultiModalDataset(train_df, captions_dict, style_to_idx, transform, base_dir=base_dir)
    val_dataset = FashionMultiModalDataset(val_df, captions_dict, style_to_idx, transform, base_dir=base_dir)
    test_dataset = FashionMultiModalDataset(test_df, captions_dict, style_to_idx, transform, base_dir=base_dir)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Compute class weights
    class_weights = compute_class_weight(
        'balanced',
        classes=np.array(list(style_to_idx.values())),
        y=train_df['style'].map(style_to_idx).values
    )
    class_weights = torch.FloatTensor(class_weights).to(device)
    
    # Initialize model (with fixed seed)
    model = MultiModalFashionClassifier(
        clip_model=clip_model,
        fashionbert_model=fashionbert_model,
        num_classes=num_classes,
        dropout=DROPOUT
    ).to(device)
    
    # Setup training
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LEARNING_RATE, 
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    
    # Training tracking
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    val_macro_f1s = []
    learning_rates = []
    
    best_val_macro_f1 = 0
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    early_stopped = False
    
    start_time = time.time()
    
    # Training loop with early stopping
    for epoch in range(MAX_EPOCHS):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc, val_preds, val_labels, val_macro_f1 = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update scheduler
        scheduler.step()
        learning_rates.append(scheduler.get_last_lr()[0])
        
        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        val_macro_f1s.append(val_macro_f1)
        
        # Track best Macro F1 (for saving & early stopping)
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), 
                      os.path.join(ARTIFACTS_DIR, "models", f"seed_{seed_value}_best_model.pth"))
        else:
            patience_counter += 1
        
        # Track best loss (for overfitting detection)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
        
        # Early stopping (based on Macro F1)
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            early_stopped = True
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOPPING_PATIENCE} epochs)")
            break
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{MAX_EPOCHS}: "
                  f"Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
                  f"Val Macro F1={val_macro_f1:.4f}")
    
    total_time = time.time() - start_time
    
    # Load best model for final evaluation
    model.load_state_dict(torch.load(
        os.path.join(ARTIFACTS_DIR, "models", f"seed_{seed_value}_best_model.pth"),
        map_location=device))
    model.eval()
    
    # Final evaluation on test set
    test_loss, test_acc, test_preds, test_labels, test_macro_f1 = validate_epoch(
        model, test_loader, criterion, device
    )
    
    # Detect overfitting
    if len(val_losses) > best_epoch:
        val_loss_after_best = min(val_losses[best_epoch:])
        overfitting_detected = val_loss_after_best > best_val_loss * 1.05
    else:
        overfitting_detected = False
    
    # Calculate train/val gap at best epoch
    train_val_gap = train_losses[best_epoch - 1] - best_val_loss if best_epoch > 0 else 0
    
    # Create learning curves plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss curves
    axes[0, 0].plot(train_losses, label='Train Loss', color='blue', linewidth=2)
    axes[0, 0].plot(val_losses, label='Val Loss', color='red', linewidth=2)
    axes[0, 0].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch {best_epoch}')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[0, 1].plot(train_accs, label='Train Acc', color='blue', linewidth=2)
    axes[0, 1].plot(val_accs, label='Val Acc', color='red', linewidth=2)
    axes[0, 1].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7)
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Macro F1 curve
    axes[1, 0].plot(val_macro_f1s, label='Val Macro F1', color='green', linewidth=2)
    axes[1, 0].axvline(x=best_epoch-1, color='red', linestyle='--', alpha=0.7)
    axes[1, 0].axhline(y=best_val_macro_f1, color='red', linestyle='--', alpha=0.7, 
                       label=f'Best: {best_val_macro_f1:.4f}')
    axes[1, 0].set_title('Validation Macro F1-Score')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Macro F1')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Summary text
    axes[1, 1].axis('off')
    summary_text = f"""
Seed: {seed_value} (Experiment {seed_idx}/{len(SEEDS)})
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Epoch: {best_epoch}
Early Stopped: {early_stopped}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Val Macro F1: {best_val_macro_f1:.4f}
Test Macro F1: {test_macro_f1:.4f}
Test Accuracy: {test_acc:.2f}%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Overfitting: {'Yes' if overfitting_detected else 'No'}
Training Time: {total_time/60:.1f} minutes
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
                    verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle(f'Learning Curves: Seed {seed_value} (Experiment {seed_idx}/{len(SEEDS)})', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(ARTIFACTS_DIR, "learning_curves", f"seed_{seed_value}_learning_curves.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Prepare results dictionary
    results = {
        "experiment_id": f"seed_{seed_value}",
        "seed_value": seed_value,
        "seed_index": seed_idx,
        "fusion": "concat_mlp",
        "seed_batch": "remaining_25",
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "learning_rate": float(LEARNING_RATE),
            "batch_size": BATCH_SIZE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "dropout": DROPOUT,
            "weight_decay": float(WEIGHT_DECAY),
            "max_epochs": MAX_EPOCHS,
            "model_init_seed": MODEL_INIT_SEED,
            "data_split_seed": seed_value,
            "dataset_size": "100% (full dataset)"
        },
        "training_info": {
            "total_epochs": len(train_losses),
            "best_epoch": best_epoch,
            "early_stopped": early_stopped,
            "total_time_minutes": float(total_time / 60)
        },
        "validation_metrics": {
            "best_val_macro_f1": float(best_val_macro_f1),
            "best_val_accuracy": float(val_accs[best_epoch - 1]) if best_epoch > 0 else 0.0,
            "best_val_loss": float(best_val_loss),
            "final_val_macro_f1": float(val_macro_f1s[-1]),
            "final_val_accuracy": float(val_accs[-1]),
            "final_val_loss": float(val_losses[-1])
        },
        "test_metrics": {
            "test_macro_f1": float(test_macro_f1),
            "test_accuracy": float(test_acc),
            "test_loss": float(test_loss)
        },
        "overfitting_analysis": {
            "overfitting_detected": overfitting_detected,
            "best_val_loss": float(best_val_loss),
            "val_loss_after_best": float(val_losses[best_epoch:][0]) if len(val_losses) > best_epoch else float(val_losses[-1]),
            "increase_percentage": float((val_losses[best_epoch:][0] - best_val_loss) / best_val_loss * 100) if len(val_losses) > best_epoch else 0.0,
            "train_val_gap": float(train_val_gap)
        },
        "training_curves": {
            "train_losses": [float(x) for x in train_losses],
            "val_losses": [float(x) for x in val_losses],
            "train_accs": [float(x) for x in train_accs],
            "val_accs": [float(x) for x in val_accs],
            "val_macro_f1s": [float(x) for x in val_macro_f1s],
            "learning_rates": [float(x) for x in learning_rates]
        },
        "data_split_info": {
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df)
        }
    }
    
    # Save results JSON
    json_path = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"  ✅ Completed: Best Val Macro F1={best_val_macro_f1:.4f}, "
          f"Test Macro F1={test_macro_f1:.4f}, "
          f"Overfitting={'Yes' if overfitting_detected else 'No'}")
    
    return results

print("✅ Experiment runner function defined!")


✅ Experiment runner function defined!


## Run 25 experiments


In [7]:
BASE_DIR = PROJECT_ROOT

all_results = []
failed_seeds = []

print("Starting", len(SEEDS), "concat+MLP runs...")
for seed_idx, seed_value in enumerate(SEEDS, 1):
    try:
        result = run_robustness_experiment(
            seed_value=seed_value,
            seed_idx=seed_idx,
            df_full=df_full,
            captions_dict=captions_dict,
            style_to_idx=style_to_idx,
            clip_model=clip_model,
            fashionbert_model=fashionbert_model,
            fashionbert_tokenizer=fashionbert_tokenizer,
            num_classes=num_classes,
            device=device,
            all_styles=all_styles,
            base_dir=BASE_DIR,
        )
        all_results.append(result)
        print("Progress:", seed_idx, "/", len(SEEDS))
    except Exception as e:
        print("Error seed", seed_value, e)
        failed_seeds.append((seed_value, str(e)))
        import traceback
        traceback.print_exc()

summary = {
    "fusion": "concat_mlp",
    "seed_batch": "remaining_25",
    "total_seeds": len(SEEDS),
    "successful_experiments": len(all_results),
    "failed_experiments": len(failed_seeds),
    "failed_seeds": [{"seed": s[0], "error": s[1]} for s in failed_seeds],
    "completed_seeds": [r["seed_value"] for r in all_results],
}
with open(os.path.join(METRICS_DIR, "experiments_summary_remaining25.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Summary:", summary["successful_experiments"], "ok,", len(failed_seeds), "failed")


Starting 25 concat+MLP runs...

Experiment 1/25 (concat+MLP): Seed 14
  Creating data splits with random_state=14...
  Split sizes: Train=9261, Val=1984, Test=1985


  Epoch 1/20: Train Loss=2.6397, Val Loss=2.6366, Val Macro F1=0.0110


  Epoch 5/20: Train Loss=2.4134, Val Loss=2.3132, Val Macro F1=0.1433


  Epoch 10/20: Train Loss=2.2112, Val Loss=2.0985, Val Macro F1=0.2068


  Epoch 15/20: Train Loss=2.1496, Val Loss=2.0461, Val Macro F1=0.2345


  Epoch 20/20: Train Loss=2.1212, Val Loss=2.0112, Val Macro F1=0.2450


  ✅ Completed: Best Val Macro F1=0.2450, Test Macro F1=0.2382, Overfitting=No
Progress: 1 / 25

Experiment 2/25 (concat+MLP): Seed 16
  Creating data splits with random_state=16...
  Split sizes: Train=9261, Val=1984, Test=1985


  Epoch 1/20: Train Loss=2.6391, Val Loss=2.6350, Val Macro F1=0.0152


KeyboardInterrupt: 